# Modin — E-commerce Business Transaction

Procesamiento distribuido del dataset de transacciones de comercio electrónico con **Modin**. Se ejecutan 10 consultas cubriendo limpieza, transformaciones, tratamiento de duplicados, filtrado, agregaciones, agrupaciones, ordenamiento y cálculo de métricas.

## Instalación de dependencias

In [4]:
!pip install -q modin[pandas] kagglehub pandas
from pathlib import Path
import modin.pandas as pd

## Descarga del dataset

Si el dataset no está disponible localmente, se descarga desde Kaggle y se deposita en `dataset/`. Las rutas son relativas a la raíz del proyecto.

In [5]:
import kagglehub

DATASET_DIR = Path.cwd().parent / "dataset"
LOCAL_FILE = DATASET_DIR / "sales_transaction.csv"

if not LOCAL_FILE.exists():
    cached = Path(kagglehub.dataset_download("gabrielramos87/an-online-shop-business"))
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE.write_bytes((cached / "Sales Transaction v.4a.csv").read_bytes())
    print(f"Dataset descargado en: {LOCAL_FILE}")
else:
    print(f"Dataset ya disponible: {LOCAL_FILE}")

Dataset ya disponible: /home/bricafio/Escritorio/BigData/dataset/sales_transaction.csv


## Lectura del dataset

In [6]:
df = pd.read_csv(LOCAL_FILE)
print(f"Dimensiones: {df.shape}")

Dimensiones: (536350, 8)


## Consulta 1: Tratar nulos de CustomerNo

Rubro: Limpieza de datos

In [7]:
nulos = df["CustomerNo"].isna().sum()
print(f"Filas con CustomerNo nulo: {nulos}")
df = df.dropna(subset=["CustomerNo"])
print(f"Dimensiones tras eliminar nulos: {df.shape}")

Filas con CustomerNo nulo: 55
Dimensiones tras eliminar nulos: (536295, 8)


## Consulta 2: Eliminar duplicados

Rubro: Deduplicación

In [8]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")
df = df.drop_duplicates(keep="first")
print(f"Dimensiones tras eliminar duplicados: {df.shape}")

Filas duplicadas: 5200
Dimensiones tras eliminar duplicados: (531095, 8)


## Consulta 3: Convertir Date y extraer componentes

Rubro: Transformación de variables

In [9]:
df["Date"] = pd.to_datetime(df["Date"])
df["Año"] = df["Date"].dt.year
df["Mes"] = df["Date"].dt.month
df["Día"] = df["Date"].dt.day
df[["Date", "Año", "Mes", "Día"]].head(3)

,Date,Año,Mes,Día
0,2019-12-09,2019,12,9
1,2019-12-09,2019,12,9
2,2019-12-09,2019,12,9


## Consulta 4: Crear TotalSales

Rubro: Transformación

In [10]:
df["TotalSales"] = df["Price"] * df["Quantity"]
df[["Price", "Quantity", "TotalSales"]].head(3)

,Price,Quantity,TotalSales
0,21.47,12,257.64
1,10.65,36,383.40
2,11.53,12,138.36


## Consulta 5: Filtrar cancelaciones

Rubro: Filtrado

In [11]:
df["TransactionNo"] = df["TransactionNo"].astype(str)
filtro = (df["Quantity"] >= 0) & (~df["TransactionNo"].str.startswith("C"))
print(f"Cancelaciones eliminadas: {(~filtro).sum()}")
df = df[filtro]
print(f"Dimensiones tras filtrar: {df.shape}")

Cancelaciones eliminadas: 8494
Dimensiones tras filtrar: (522601, 12)


## Consulta 6: Facturación mensual

Rubro: Agrupación y agregación

In [12]:
(df.groupby(["Año", "Mes"])["TotalSales"]
 .sum().reset_index().sort_values(["Año", "Mes"]))

,Año,Mes,TotalSales
0,2018,12,4397648.39
1,2019,1,4548423.47
2,2019,2,3327342.64
3,2019,3,4384669.82
4,2019,4,3579310.06
5,2019,5,4569952.21
6,2019,6,4486050.15
7,2019,7,4571494.88
8,2019,8,4749801.23
9,2019,9,6613772.79


## Consulta 7: Ingreso por país

Rubro: Agregación

In [13]:
(df.groupby("Country")["TotalSales"]
 .sum().reset_index().sort_values("TotalSales", ascending=False)
 .rename(columns={"TotalSales": "IngresoTotal"}))

,Country,IngresoTotal
36,United Kingdom,52346795.60
24,Netherlands,2151553.59
10,EIRE,1711819.39
14,Germany,1369839.62
13,France,1329903.39
0,Australia,995414.01
32,Sweden,401879.89
33,Switzerland,361691.96
20,Japan,293155.44
31,Spain,280843.80


## Consulta 8: Top 10 productos

Rubro: Agrupación y ordenamiento

In [14]:
(df.groupby(["ProductNo", "ProductName"])["Quantity"]
 .sum().reset_index().sort_values("Quantity", ascending=False)
 .rename(columns={"Quantity": "Unidades"}).head(10))

,ProductNo,ProductName,Unidades
2446,23843,Paper Craft Little Birdie,80995
2004,23166,Medium Ceramic Top Storage Jar,78033
1094,22197,Popcorn Holder,56902
2840,84077,World War 2 Gliders Asstd Designs,54951
3256,85099B,Jumbo Bag Red Retrospot,48375
3271,85123A,Cream Hanging Heart T-Light Holder,37937
426,21212,Pack Of 72 Retrospot Cake Cases,36492
3095,84879,Assorted Colour Bird Ornament,36394
1926,23084,Rabbit Night Light,30742
1359,22492,Mini Paint Set Vintage,26633


## Consulta 9: Top 10 clientes

Rubro: Agrupación, agregación y ordenamiento

In [15]:
(df.groupby("CustomerNo")["TotalSales"]
 .sum().reset_index().sort_values("TotalSales", ascending=False)
 .rename(columns={"TotalSales": "GastoAcumulado"}).head(10))

,CustomerNo,GastoAcumulado
1880,14646.0,2112282.03
3302,16446.0,1002741.57
2085,14911.0,914204.19
126,12415.0,900545.54
4581,18102.0,897137.36
4082,17450.0,891069.53
68,12346.0,840113.80
1506,14156.0,694202.51
1140,13694.0,646116.78
4127,17511.0,639006.19


## Consulta 10: Estadísticos descriptivos

Rubro: Cálculo de métricas

In [16]:
df[["Price", "Quantity", "TotalSales"]].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Price,522601.0,12.637160,7.965974,5.13,10.99,11.94,14.09,660.62
Quantity,522601.0,10.667492,157.542420,1.00,1.00,4.00,12.00,80995.00
TotalSales,522601.0,120.132385,1860.158603,5.13,17.90,44.48,120.80,1002718.10
